Task 1: Install Dependencies and Run Spark Session

In [1]:
! pip install pyspark

In [2]:
from pyspark.sql import SparkSession

In [3]:
spark = SparkSession.builder.appName("ML").getOrCreate()

In [4]:
spark

Task 2: Clone and Explore Dataset

In [5]:
! git clone https://github.com/education454/diabetes_dataset

Cloning into 'diabetes_dataset'...
remote: Enumerating objects: 6, done.
remote: Counting objects: 100% (6/6), done.
remote: Compressing objects: 100% (5/5), done.
remote: Total 6 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (6/6), 13.02 KiB | 6.51 MiB/s, done.


In [6]:
! ls diabetes_dataset

diabetes.csv  new_test.csv


In [7]:
sdf = spark.read.csv("/content/diabetes_dataset/diabetes.csv", header = True, inferSchema = True)

In [9]:
sdf.show(5)
# 1 : patient has diabetes
# 0: no diabetes

+-----------+-------+-------------+-------------+-------+----+------------------------+---+-------+
|Pregnancies|Glucose|BloodPressure|SkinThickness|Insulin| BMI|DiabetesPedigreeFunction|Age|Outcome|
+-----------+-------+-------------+-------------+-------+----+------------------------+---+-------+
|          2|    138|           62|           35|      0|33.6|                   0.127| 47|      1|
|          0|     84|           82|           31|    125|38.2|                   0.233| 23|      0|
|          0|    145|            0|            0|      0|44.2|                    0.63| 31|      1|
|          0|    135|           68|           42|    250|42.3|                   0.365| 24|      1|
|          1|    139|           62|           41|    480|40.7|                   0.536| 21|      0|
+-----------+-------+-------------+-------------+-------+----+------------------------+---+-------+
only showing top 5 rows


In [10]:
sdf.count()

2000

In [11]:
sdf.printSchema()

root
 |-- Pregnancies: integer (nullable = true)
 |-- Glucose: integer (nullable = true)
 |-- BloodPressure: integer (nullable = true)
 |-- SkinThickness: integer (nullable = true)
 |-- Insulin: integer (nullable = true)
 |-- BMI: double (nullable = true)
 |-- DiabetesPedigreeFunction: double (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Outcome: integer (nullable = true)



In [12]:
len(sdf.columns)

9

In [13]:
sdf.groupBy("outcome").count().show()

+-------+-----+
|outcome|count|
+-------+-----+
|      1|  684|
|      0| 1316|
+-------+-----+



In [14]:
sdf.describe().show()

+-------+-----------------+------------------+------------------+-----------------+-----------------+------------------+------------------------+------------------+------------------+
|summary|      Pregnancies|           Glucose|     BloodPressure|    SkinThickness|          Insulin|               BMI|DiabetesPedigreeFunction|               Age|           Outcome|
+-------+-----------------+------------------+------------------+-----------------+-----------------+------------------+------------------------+------------------+------------------+
|  count|             2000|              2000|              2000|             2000|             2000|              2000|                    2000|              2000|              2000|
|   mean|           3.7035|          121.1825|           69.1455|           20.935|           80.254|32.192999999999984|     0.47092999999999974|           33.0905|             0.342|
| stddev|3.306063032730656|32.068635649902916|19.188314815604098|16.103242909926

Task 3: Data Cleaning and Preparation

In [15]:
sdf.columns

['Pregnancies',
 'Glucose',
 'BloodPressure',
 'SkinThickness',
 'Insulin',
 'BMI',
 'DiabetesPedigreeFunction',
 'Age',
 'Outcome']

In [16]:
sdf[sdf['Glucose'].isNull()].count()

0

In [17]:
for col in sdf.columns:
  print(col+":", sdf[sdf[col].isNull()].count())

Pregnancies: 0
Glucose: 0
BloodPressure: 0
SkinThickness: 0
Insulin: 0
BMI: 0
DiabetesPedigreeFunction: 0
Age: 0
Outcome: 0


In [22]:
def count_zeros():
  columns_list = sdf.columns
  for i in columns_list:
    print(i+":", sdf[sdf[i]==0].count())

In [23]:
count_zeros()

Pregnancies: 301
Glucose: 13
BloodPressure: 90
SkinThickness: 573
Insulin: 956
BMI: 28
DiabetesPedigreeFunction: 0
Age: 0
Outcome: 1316


In [19]:
sdf.show(5)

+-----------+-------+-------------+-------------+-------+----+------------------------+---+-------+
|Pregnancies|Glucose|BloodPressure|SkinThickness|Insulin| BMI|DiabetesPedigreeFunction|Age|Outcome|
+-----------+-------+-------------+-------------+-------+----+------------------------+---+-------+
|          2|    138|           62|           35|      0|33.6|                   0.127| 47|      1|
|          0|     84|           82|           31|    125|38.2|                   0.233| 23|      0|
|          0|    145|            0|            0|      0|44.2|                    0.63| 31|      1|
|          0|    135|           68|           42|    250|42.3|                   0.365| 24|      1|
|          1|    139|           62|           41|    480|40.7|                   0.536| 21|      0|
+-----------+-------+-------------+-------------+-------+----+------------------------+---+-------+
only showing top 5 rows


In [24]:
def count_zeros():
  columns_list = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]
  for i in columns_list:
    print(i+":", sdf[sdf[i]==0].count())

In [25]:
count_zeros()

Glucose: 13
BloodPressure: 90
SkinThickness: 573
Insulin: 956
BMI: 28


In [26]:
from pyspark.sql.functions import *
for i in sdf.columns[1:6]:
  data = sdf.agg({i:"mean"}).first()[0]
  print("mean value for {} is {}".format(i,int(data)))
  sdf =sdf.withColumn(i, when(sdf[i]==0, int(data)).otherwise(sdf[i]))

mean value for Glucose is 121
mean value for BloodPressure is 69
mean value for SkinThickness is 20
mean value for Insulin is 80
mean value for BMI is 32


In [28]:
sdf.show(5)

+-----------+-------+-------------+-------------+-------+----+------------------------+---+-------+
|Pregnancies|Glucose|BloodPressure|SkinThickness|Insulin| BMI|DiabetesPedigreeFunction|Age|Outcome|
+-----------+-------+-------------+-------------+-------+----+------------------------+---+-------+
|          2|    138|           62|           35|     80|33.6|                   0.127| 47|      1|
|          0|     84|           82|           31|    125|38.2|                   0.233| 23|      0|
|          0|    145|           69|           20|     80|44.2|                    0.63| 31|      1|
|          0|    135|           68|           42|    250|42.3|                   0.365| 24|      1|
|          1|    139|           62|           41|    480|40.7|                   0.536| 21|      0|
+-----------+-------+-------------+-------------+-------+----+------------------------+---+-------+
only showing top 5 rows


In [29]:
count_zeros()

Glucose: 0
BloodPressure: 0
SkinThickness: 0
Insulin: 0
BMI: 0


Task 4: Correlation Analysis and Feature Selection

In [30]:
for i in sdf.columns:
  print("Correlation to outocme for {} is {}".format(i, sdf.stat.corr("Outcome",i)))

Correlation to outocme for Pregnancies is 0.22443699263363961
Correlation to outocme for Glucose is 0.48796646527321064
Correlation to outocme for BloodPressure is 0.17171333286446713
Correlation to outocme for SkinThickness is 0.1659010662889893
Correlation to outocme for Insulin is 0.1711763270226193
Correlation to outocme for BMI is 0.2827927569760082
Correlation to outocme for DiabetesPedigreeFunction is 0.1554590791569403
Correlation to outocme for Age is 0.23650924717620253
Correlation to outocme for Outcome is 1.0


In [33]:
from pyspark.ml.feature import VectorAssembler

In [34]:
sdf.show(5)

+-----------+-------+-------------+-------------+-------+----+------------------------+---+-------+
|Pregnancies|Glucose|BloodPressure|SkinThickness|Insulin| BMI|DiabetesPedigreeFunction|Age|Outcome|
+-----------+-------+-------------+-------------+-------+----+------------------------+---+-------+
|          2|    138|           62|           35|     80|33.6|                   0.127| 47|      1|
|          0|     84|           82|           31|    125|38.2|                   0.233| 23|      0|
|          0|    145|           69|           20|     80|44.2|                    0.63| 31|      1|
|          0|    135|           68|           42|    250|42.3|                   0.365| 24|      1|
|          1|    139|           62|           41|    480|40.7|                   0.536| 21|      0|
+-----------+-------+-------------+-------------+-------+----+------------------------+---+-------+
only showing top 5 rows


In [35]:
assembler = VectorAssembler(inputCols = ["Pregnancies", "Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI", "DiabetesPedigreeFunction", "Age"], outputCol = "features")
output_data = assembler.transform(sdf)

In [36]:
output_data.printSchema()

root
 |-- Pregnancies: integer (nullable = true)
 |-- Glucose: integer (nullable = true)
 |-- BloodPressure: integer (nullable = true)
 |-- SkinThickness: integer (nullable = true)
 |-- Insulin: integer (nullable = true)
 |-- BMI: double (nullable = true)
 |-- DiabetesPedigreeFunction: double (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Outcome: integer (nullable = true)
 |-- features: vector (nullable = true)



In [37]:
output_data.show()

+-----------+-------+-------------+-------------+-------+----+------------------------+---+-------+--------------------+
|Pregnancies|Glucose|BloodPressure|SkinThickness|Insulin| BMI|DiabetesPedigreeFunction|Age|Outcome|            features|
+-----------+-------+-------------+-------------+-------+----+------------------------+---+-------+--------------------+
|          2|    138|           62|           35|     80|33.6|                   0.127| 47|      1|[2.0,138.0,62.0,3...|
|          0|     84|           82|           31|    125|38.2|                   0.233| 23|      0|[0.0,84.0,82.0,31...|
|          0|    145|           69|           20|     80|44.2|                    0.63| 31|      1|[0.0,145.0,69.0,2...|
|          0|    135|           68|           42|    250|42.3|                   0.365| 24|      1|[0.0,135.0,68.0,4...|
|          1|    139|           62|           41|    480|40.7|                   0.536| 21|      0|[1.0,139.0,62.0,4...|
|          0|    173|           

Task 5: Split Dataset and Build Model

In [39]:
from typing_extensions import final
from pyspark.ml.classification import LogisticRegression
final_data = output_data.select("features", "Outcome")

In [40]:
final_data.printSchema()

root
 |-- features: vector (nullable = true)
 |-- Outcome: integer (nullable = true)



In [41]:
train, test = final_data.randomSplit([0.7, 0.3])
models = LogisticRegression(labelCol = "Outcome")
model = models.fit(train)

In [42]:
summary = model.summary
summary.predictions.describe().show()

+-------+------------------+-------------------+
|summary|           Outcome|         prediction|
+-------+------------------+-------------------+
|  count|              1398|               1398|
|   mean|0.3354792560801145|0.24892703862660945|
| stddev|0.4723266926398707| 0.4325461818086445|
|    min|               0.0|                0.0|
|    max|               1.0|                1.0|
+-------+------------------+-------------------+



In [43]:
summary.predictions.show(20)

+--------------------+-------+--------------------+--------------------+----------+
|            features|Outcome|       rawPrediction|         probability|prediction|
+--------------------+-------+--------------------+--------------------+----------+
|[0.0,57.0,60.0,20...|    0.0|[4.19205323083376...|[0.98510984990549...|       0.0|
|[0.0,57.0,60.0,20...|    0.0|[4.19205323083376...|[0.98510984990549...|       0.0|
|[0.0,67.0,76.0,20...|    0.0|[2.41501607917035...|[0.91796521093770...|       0.0|
|[0.0,67.0,76.0,20...|    0.0|[2.41501607917035...|[0.91796521093770...|       0.0|
|[0.0,74.0,52.0,10...|    0.0|[3.62014086009698...|[0.97391950298185...|       0.0|
|[0.0,78.0,88.0,29...|    0.0|[2.60705418160806...|[0.93131419723518...|       0.0|
|[0.0,78.0,88.0,29...|    0.0|[2.60705418160806...|[0.93131419723518...|       0.0|
|[0.0,84.0,64.0,22...|    0.0|[2.43902048416190...|[0.91975482350853...|       0.0|
|[0.0,84.0,64.0,22...|    0.0|[2.43902048416190...|[0.91975482350853...|    

In [48]:
summary.predictions.show(20, truncate=False)

+------------------------------------------+-------+----------------------------------------+-----------------------------------------+----------+
|features                                  |Outcome|rawPrediction                           |probability                              |prediction|
+------------------------------------------+-------+----------------------------------------+-----------------------------------------+----------+
|[0.0,57.0,60.0,20.0,80.0,21.7,0.735,67.0] |0.0    |[4.192053230833766,-4.192053230833766]  |[0.9851098499054987,0.014890150094501342]|0.0       |
|[0.0,57.0,60.0,20.0,80.0,21.7,0.735,67.0] |0.0    |[4.192053230833766,-4.192053230833766]  |[0.9851098499054987,0.014890150094501342]|0.0       |
|[0.0,67.0,76.0,20.0,80.0,45.3,0.194,46.0] |0.0    |[2.415016079170356,-2.415016079170356]  |[0.9179652109377047,0.08203478906229533] |0.0       |
|[0.0,67.0,76.0,20.0,80.0,45.3,0.194,46.0] |0.0    |[2.415016079170356,-2.415016079170356]  |[0.9179652109377047,0.082

Task 6: Evaluate and Save Model

In [51]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator
predictions = model.evaluate(test)

In [52]:
predictions.predictions.show(20, truncate=False)

+------------------------------------------+-------+----------------------------------------+-----------------------------------------+----------+
|features                                  |Outcome|rawPrediction                           |probability                              |prediction|
+------------------------------------------+-------+----------------------------------------+-----------------------------------------+----------+
|[0.0,73.0,69.0,20.0,80.0,21.1,0.342,25.0] |0      |[4.235674162913012,-4.235674162913012]  |[0.9857363441022887,0.01426365589771128] |0.0       |
|[0.0,73.0,69.0,20.0,80.0,21.1,0.342,25.0] |0      |[4.235674162913012,-4.235674162913012]  |[0.9857363441022887,0.01426365589771128] |0.0       |
|[0.0,74.0,52.0,10.0,36.0,27.8,0.269,22.0] |0      |[3.6201408600969813,-3.6201408600969813]|[0.9739195029818564,0.026080497018143634]|0.0       |
|[0.0,74.0,52.0,10.0,36.0,27.8,0.269,22.0] |0      |[3.6201408600969813,-3.6201408600969813]|[0.9739195029818564,0.026

In [53]:
evaluator = BinaryClassificationEvaluator(rawPredictionCol = "rawPrediction", labelCol = "Outcome")
evaluator.evaluate(model.transform(test))

0.852364641547984

In [54]:
model.save("model3")

In [55]:
# load saved model back to the envrionemnt
from  pyspark.ml.classification import LogisticRegressionModel
model = LogisticRegressionModel.load("model3")

Task 7: Prediction on New Data with the saved model

In [56]:
test_df = spark.read.csv("/content/diabetes_dataset/new_test.csv", header = True, inferSchema = True)

In [57]:
test_df.printSchema()

root
 |-- Pregnancies: integer (nullable = true)
 |-- Glucose: integer (nullable = true)
 |-- BloodPressure: integer (nullable = true)
 |-- SkinThickness: integer (nullable = true)
 |-- Insulin: integer (nullable = true)
 |-- BMI: double (nullable = true)
 |-- DiabetesPedigreeFunction: double (nullable = true)
 |-- Age: integer (nullable = true)



In [58]:
test_df.show()

+-----------+-------+-------------+-------------+-------+----+------------------------+---+
|Pregnancies|Glucose|BloodPressure|SkinThickness|Insulin| BMI|DiabetesPedigreeFunction|Age|
+-----------+-------+-------------+-------------+-------+----+------------------------+---+
|          1|    190|           78|           38|    150|45.1|                   0.153| 48|
|          0|     80|           84|           36|    120|50.2|                   0.211| 26|
|          2|    138|           82|           46|    255|52.3|                   0.315| 30|
|          1|    110|           63|           44|    480|62.7|                   0.616| 32|
+-----------+-------+-------------+-------------+-------+----+------------------------+---+



In [59]:
test_data = assembler.transform(test_df)

In [60]:
test_data.printSchema()

root
 |-- Pregnancies: integer (nullable = true)
 |-- Glucose: integer (nullable = true)
 |-- BloodPressure: integer (nullable = true)
 |-- SkinThickness: integer (nullable = true)
 |-- Insulin: integer (nullable = true)
 |-- BMI: double (nullable = true)
 |-- DiabetesPedigreeFunction: double (nullable = true)
 |-- Age: integer (nullable = true)
 |-- features: vector (nullable = true)



In [61]:
results = model.transform(test_data)
results.printSchema()

root
 |-- Pregnancies: integer (nullable = true)
 |-- Glucose: integer (nullable = true)
 |-- BloodPressure: integer (nullable = true)
 |-- SkinThickness: integer (nullable = true)
 |-- Insulin: integer (nullable = true)
 |-- BMI: double (nullable = true)
 |-- DiabetesPedigreeFunction: double (nullable = true)
 |-- Age: integer (nullable = true)
 |-- features: vector (nullable = true)
 |-- rawPrediction: vector (nullable = true)
 |-- probability: vector (nullable = true)
 |-- prediction: double (nullable = false)



In [62]:
results.select("features", "prediction").show()

+--------------------+----------+
|            features|prediction|
+--------------------+----------+
|[1.0,190.0,78.0,3...|       1.0|
|[0.0,80.0,84.0,36...|       0.0|
|[2.0,138.0,82.0,4...|       1.0|
|[1.0,110.0,63.0,4...|       1.0|
+--------------------+----------+

